# M1 Colab CUDA component-precision audit

Select **Runtime → Change runtime type → T4 GPU** (free Colab) or **L4 GPU** before running. This notebook clones the integrated `Kaushal-Commits` branch, installs the GPU dependencies, and runs every requested FP16/NF4 candidate in a fresh subprocess. Each audit verifies actual module classes and quantization state, measures unique weight and metadata storage, runs one synthetic inference, and records peak allocated/reserved GPU memory.

This is an M1 feasibility audit, not an accuracy experiment. Generated JSON artifacts remain outside Git and should be downloaded before the Colab session ends.

In [1]:
import subprocess

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
print(gpu_check.stdout.strip())
gpu_name = gpu_check.stdout.splitlines()[0].split(",", 1)[0].strip()
SUPPORTED_GPUS = ("L4", "T4")
if not any(name in gpu_name for name in SUPPORTED_GPUS):
    raise RuntimeError(f"Expected a Colab L4 or T4 GPU; found {gpu_name!r}.")
GPU_TAG = next(name for name in SUPPORTED_GPUS if name in gpu_name).lower()

Tesla T4, 15360 MiB, 580.82.07


In [2]:
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/Omkar38/mixed-precision-visual-shift.git"
BRANCH = "Kaushal-Commits"
REPO_DIR = Path("/content/mixed-precision-visual-shift")
if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone. Use a fresh Colab runtime.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Checked out {BRANCH} at {commit}")
required_paths = [REPO_DIR / "pyproject.toml", REPO_DIR / "src/mpvs/cli.py"]
missing = [str(path.relative_to(REPO_DIR)) for path in required_paths if not path.is_file()]
if missing:
    raise RuntimeError(
        f"Commit {commit} does not contain the published M1 implementation; missing {missing}. "
        f"Commit and push the local changes to origin/{BRANCH}, then rerun this cell."
    )
%cd /content/mixed-precision-visual-shift

Checked out Kaushal-Commits at 8d9fee56367bccb1c27bce67c88f8024e53c8dfd


RuntimeError: Commit 8d9fee56367bccb1c27bce67c88f8024e53c8dfd does not contain the published M1 implementation; missing ['pyproject.toml']. Commit and push the local changes to origin/Kaushal-Commits, then rerun this cell.

In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-enabled PyTorch; pin versions from the successful T4 FP16 smoke test.
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "transformers==4.52.4", "accelerate==1.15.0", "bitsandbytes==0.50.2",
    "Pillow", "sentencepiece==0.2.2", "num2words==0.5.14",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

In [ ]:
from pathlib import Path
import subprocess
import sys

Path("artifacts").mkdir(exist_ok=True)
subprocess.run([
    sys.executable, "scripts/preflight.py",
    "--output", f"artifacts/preflight-colab-{GPU_TAG}.json",
], check=True)

In [ ]:
import importlib.metadata
import torch
import bitsandbytes

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime and restart.")
package_versions = {
    name: importlib.metadata.version(name)
    for name in ("torch", "transformers", "accelerate", "bitsandbytes", "Pillow", "sentencepiece", "num2words")
}
print({"gpu": torch.cuda.get_device_name(0), "torch_cuda": torch.version.cuda, "packages": package_versions})

## Run isolated candidate audits

The first three entries are the feasibility gate: FP16, uniform NF4, and language-only NF4. `RUN_FULL_GRID=True` continues through all eight requested allocations. A failed candidate is retained as a diagnostic JSON record and does not become a model answer.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

FEASIBILITY_GATE = [
    "v-fp16_c-fp16_l-fp16",
    "v-nf4_c-nf4_l-nf4",
    "v-fp16_c-fp16_l-nf4",
]
ALL_CANDIDATES = [
    "v-fp16_c-fp16_l-fp16", "v-fp16_c-fp16_l-nf4",
    "v-fp16_c-nf4_l-fp16", "v-fp16_c-nf4_l-nf4",
    "v-nf4_c-fp16_l-fp16", "v-nf4_c-fp16_l-nf4",
    "v-nf4_c-nf4_l-fp16", "v-nf4_c-nf4_l-nf4",
]
RUN_FULL_GRID = True
candidate_ids = FEASIBILITY_GATE + [c for c in ALL_CANDIDATES if c not in FEASIBILITY_GATE] if RUN_FULL_GRID else FEASIBILITY_GATE
results = []
environment = {**os.environ, "PYTHONUNBUFFERED": "1"}
for candidate_id in candidate_ids:
    output = Path("artifacts") / f"m1-audit-{GPU_TAG}-{candidate_id}.json"
    print(f"\n=== {candidate_id} ===")
    completed = subprocess.run([
        sys.executable, "-m", "mpvs.cli", "audit",
        "--candidate", candidate_id, "--device", "cuda",
        "--smoke-inference", "--output", str(output),
    ], text=True, capture_output=True, env=environment)
    if completed.stderr:
        print(completed.stderr[-2000:])
    record = json.loads(output.read_text()) if output.exists() else {"failure_reason": "audit JSON was not written"}
    result = {
        "candidate_id": candidate_id, "returncode": completed.returncode,
        "load_status": record.get("load_status"),
        "configuration_status": record.get("configuration_status"),
        "inference_status": record.get("inference_status"),
        "weight_bytes": record.get("memory", {}).get("total_weight_storage_bytes"),
        "failure_reason": record.get("failure_reason"),
    }
    results.append(result)
    print(result)
print(json.dumps(results, indent=2))

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

summary_path = Path("artifacts") / f"m1-audit-{GPU_TAG}-summary.json"
audit_paths = [
    path for path in sorted(Path("artifacts").glob(f"m1-audit-{GPU_TAG}-*.json"))
    if path != summary_path
]
completed = subprocess.run([
    sys.executable, "-m", "mpvs.cli", "summarize",
    *map(str, audit_paths), "--output", str(summary_path),
], text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
summary = json.loads(summary_path.read_text())
summary["repository_commit"] = commit
summary["gpu_name"] = gpu_name
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
print(json.dumps(summary, indent=2))
print(f"Summary exit code: {completed.returncode}; acceptance={summary['m1_candidate_acceptance_met']}")

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(f"m1-audit-{GPU_TAG}-{commit[:8]}", "zip", "artifacts")
print(f"Downloading {archive}")
files.download(archive)